In [14]:
import sys
import os

sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model # Calculating the GARCH model
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
print("All imports ready!")

All imports ready!


In [15]:

from src.data_loader import SP500DataLoader

# Load data covering COVID period
loader_covid = SP500DataLoader(start_year=2005, end_year=2022)
df_covid = loader_covid.load_all()

# Extract pre-COVID (normal) period: 2010-2019
pre_covid_returns = df_covid['2010':'2019']['log_returns'].dropna() * 100

# Extract COVID crisis period: 2020-2021
covid_crisis_returns = df_covid['2020':'2021']['log_returns'].dropna() * 100

print(f"Data loaded: {df_covid.index[0]} to {df_covid.index[-1]}")
print(f"Shape: {df_covid.shape}")
print(f"\nFirst 3 rows:")
print(df_covid.head(3))

Data loaded: 2005-01-03 00:00:00 to 2022-12-30 00:00:00
Shape: (4531, 8)

First 3 rows:
              adj_close        close         high          low         open  \
Date                                                                          
2005-01-03  1202.079956  1202.079956  1217.800049  1200.319946  1211.920044   
2005-01-04  1188.050049  1188.050049  1205.839966  1185.390015  1202.079956   
2005-01-05  1183.739990  1183.739990  1192.729980  1183.719971  1188.050049   

                volume   returns  log_returns  
Date                                           
2005-01-03  1510800000       NaN          NaN  
2005-01-04  1721000000 -0.011671    -0.011740  
2005-01-05  1738900000 -0.003628    -0.003634  


In [16]:
print(f"Pre-COVID returns: {len(pre_covid_returns)} days")
print(f"  From: {pre_covid_returns.index[0]} to {pre_covid_returns.index[-1]}")
print(f"  Mean return: {pre_covid_returns.mean():.4f}%")

print(f"\nCOVID crisis returns: {len(covid_crisis_returns)} days")
print(f"  From: {covid_crisis_returns.index[0]} to {covid_crisis_returns.index[-1]}")
print(f"  Mean return: {covid_crisis_returns.mean():.4f}%")

Pre-COVID returns: 2516 days
  From: 2010-01-04 00:00:00 to 2019-12-31 00:00:00
  Mean return: 0.0423%

COVID crisis returns: 505 days
  From: 2020-01-02 00:00:00 to 2021-12-31 00:00:00
  Mean return: 0.0770%


In [17]:
def fit_gjr_garch(returns, period_name="Model"):
    """
    Fit GJR-GARCH(1,1) model and return parameters.
    """
    print(f"\n{'='*70}")
    print(f"FITTING GJR-GARCH: {period_name}")
    print(f"{'='*70}")
    print(f"Data points: {len(returns)}")
    print(f"Date range: {returns.index[0]} to {returns.index[-1]}")
    print(f"Mean return: {returns.mean():.6f}%")
    print(f"Std deviation: {returns.std():.6f}%")



    # Scale returns for numerical stability
    returns_scaled = returns *10

    
    # Fit GJR-GARCH(1,1) with asymmetry term (o=1)
    model = arch_model(
        returns, 
        vol='GARCH', 
        p=1, 
        q=1, 
        o=1,               # Asymmetry term for leverage effect
        dist='normal'
    )
    
    results = model.fit(update_freq=5, disp='off')
    
    # Extract parameters
    params = {
        'omega': results.params['omega'],
        'alpha': results.params['alpha[1]'],
        'gamma': results.params['gamma[1]'],
        'beta': results.params['beta[1]'],
        'persistence': results.params['alpha[1]'] + results.params['beta[1]'] + results.params['gamma[1]']/2,
        'p_value_gamma': results.pvalues['gamma[1]'],
        'log_likelihood': results.loglikelihood,
        'aic': results.aic,
        'bic': results.bic
    }
    
    # Print key results
    print(f"\n--- PARAMETERS ---")
    print(f"ω (omega) = {params['omega']:.6f}  (baseline volatility)")
    print(f"α (alpha) = {params['alpha']:.4f}  (reaction to past shock)")
    print(f"γ (gamma) = {params['gamma']:.4f}  (EXTRA reaction to BAD news) ← LEVERAGE EFFECT")
    print(f"β (beta)  = {params['beta']:.4f}  (volatility persistence)")
    print(f"\nPersistence = {params['persistence']:.4f} (should be < 1)")
    
    # Check leverage effect
    if params['gamma'] > 0 and params['p_value_gamma'] < 0.05:
        print(f"\n LEVERAGE EFFECT CONFIRMED: γ = {params['gamma']:.4f} > 0, p = {params['p_value_gamma']:.4f} < 0.05")
    elif params['gamma'] > 0:
        print(f"\n Leverage effect present but not significant (p = {params['p_value_gamma']:.4f})")
    else:
        print(f"\n No leverage effect detected (γ = {params['gamma']:.4f})")
    
    return params, results

In [18]:
# Fit GARCH on pre-COVID period (2010-2019)
params_pre_covid, results_pre_covid = fit_gjr_garch(pre_covid_returns, "PRE-COVID (2010-2019)")


FITTING GJR-GARCH: PRE-COVID (2010-2019)
Data points: 2516
Date range: 2010-01-04 00:00:00 to 2019-12-31 00:00:00
Mean return: 0.042281%
Std deviation: 0.932066%

--- PARAMETERS ---
ω (omega) = 0.035730  (baseline volatility)
α (alpha) = 0.0000  (reaction to past shock)
γ (gamma) = 0.2793  (EXTRA reaction to BAD news) ← LEVERAGE EFFECT
β (beta)  = 0.8161  (volatility persistence)

Persistence = 0.9558 (should be < 1)

 LEVERAGE EFFECT CONFIRMED: γ = 0.2793 > 0, p = 0.0000 < 0.05


In [20]:
# Fit GARCH on COVID crisis period (2020-2021)
params_covid, results_covid = fit_gjr_garch(covid_crisis_returns, "COVID CRISIS (2020-2021)")


FITTING GJR-GARCH: COVID CRISIS (2020-2021)
Data points: 505
Date range: 2020-01-02 00:00:00 to 2021-12-31 00:00:00
Mean return: 0.076994%
Std deviation: 1.651391%

--- PARAMETERS ---
ω (omega) = 0.114953  (baseline volatility)
α (alpha) = 0.1762  (reaction to past shock)
γ (gamma) = 0.3328  (EXTRA reaction to BAD news) ← LEVERAGE EFFECT
β (beta)  = 0.6238  (volatility persistence)

Persistence = 0.9664 (should be < 1)

 Leverage effect present but not significant (p = 0.1336)
